# seed36 — Val RT-DETR50 (raw vs curated) trên `data/raw` val

Đánh giá 2 checkpoint `.pt` của **raw_rtdetr50** và **curated_rtdetr50** (seed36) trên split **val** của dataset `data/raw` (2272 ảnh, 6 class).

- Checkpoint là state-dict của Hugging Face `RTDetrForObjectDetection` (`PekingU/rtdetr_r50vd`). Nếu là checkpoint Ultralytics, notebook tự chuyển sang `RTDETR(...).val()`.
- Metric giống `src/evaluator.py`: Precision / Recall / F1 / mAP50 / mAP50-95 / **MEAN_5** (AP50 trung bình 5 class lỗi, bỏ `insulator`), cộng với AP theo từng class.
- Kết quả được ghi vào `<SEED_DIR>/val_raw_results/`.

## 1. Cấu hình — chỉnh ở đây nếu tự dò sai

In [ ]:
from pathlib import Path
import os

# Thư mục seed36. Trên Camber Jupyter, stash://bunpmc/ được mount ở home
# => stash://bunpmc/projects/finding_seed/seed36/  ->  ~/projects/finding_seed/seed36
SEED_DIR_CANDIDATES = [
    Path.cwd(),
    Path.home() / "projects/finding_seed/seed36",
    Path.home() / "stash/projects/finding_seed/seed36",
]

# Dataset data/raw (YOLO format: images/val + labels/val)
DATA_ROOT_CANDIDATES = [
    Path.cwd() / "data/raw",
    Path.home() / "projects/finding_seed/seed36/data/raw",
    Path.home() / "projects/finding_seed/data/raw",
    Path.home() / "datasets/raw",
    Path.home() / "datasets/v6p5_roboflow_ablation",
    Path(r"C:\Users\intern\Documents\SEP490\train\benchmakring_transmission_line_defect_model\data\raw"),
]
SPLIT = "val"

# Đường dẫn tương đối so với SEED_DIR (stash://bunpmc/projects/finding_seed/seed36/...)
# Để None thì tự dò trong SEED_DIR
RUNS = {
    "raw_rtdetr50": "raw_rtdetr50/weights/best.pt",
    "curated_rtdetr50": "curated_rtdetr50/weights/best.pt",
}

BATCH_SIZE = 8          # giảm xuống 2-4 nếu OOM
DEVICE = "cuda:0"       # tự fallback sang CPU nếu không có GPU
MAX_IMAGES = None       # đặt số nhỏ (vd 50) để chạy thử nhanh

CLASS_NAMES = {0: "insulator", 1: "insulator_broken", 2: "insulator_flashover",
               3: "bird_nest", 4: "foreign_object", 5: "broken_strand"}
DEFECT_CLASSES = ["insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand"]

## 2. Môi trường

In [ ]:
import sys, subprocess, importlib
for pkg, pip_name in [("transformers", "transformers"), ("ultralytics", "ultralytics"), ("timm", "timm")]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import re, time, json
import numpy as np, pandas as pd, torch
from PIL import Image
import transformers, ultralytics
print("torch", torch.__version__, "| transformers", transformers.__version__, "| ultralytics", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    DEVICE = "cpu"

## 3. Tìm seed36, dataset và checkpoint

In [ ]:
def first_existing(cands, check):
    for c in cands:
        try:
            if check(c):
                return c.resolve()
        except OSError:
            pass
    return None

SEED_DIR = first_existing(SEED_DIR_CANDIDATES, lambda p: p.is_dir() and any(p.rglob("*.pt")))
assert SEED_DIR, f"Không thấy thư mục seed36 có file .pt trong: {SEED_DIR_CANDIDATES}"
DATA_ROOT = first_existing(DATA_ROOT_CANDIDATES, lambda p: (p / "images" / SPLIT).is_dir() and (p / "labels" / SPLIT).is_dir())
assert DATA_ROOT, f"Không thấy data/raw (images/{SPLIT} + labels/{SPLIT}) trong: {DATA_ROOT_CANDIDATES}"
print("SEED_DIR :", SEED_DIR)
print("DATA_ROOT:", DATA_ROOT)

all_pt = sorted(SEED_DIR.rglob("*.pt"))
print(f"\n{len(all_pt)} file .pt trong seed:")
for p in all_pt:
    print(f"  {p.stat().st_size/1e6:8.1f} MB  {p.relative_to(SEED_DIR)}")

def pick_checkpoint(run_name):
    dataset_tag, _ = run_name.split("_", 1)            # raw / curated
    def matches(p):
        s = str(p.relative_to(SEED_DIR)).lower().replace("\\", "/")
        is_r50 = re.search(r"rtdetr_?r?50", s) is not None
        return is_r50 and dataset_tag in s and not (dataset_tag == "raw" and "curated" in s)
    cands = [p for p in all_pt if matches(p)]
    def rank(p):
        n = p.name.lower()
        return (0 if "best" in n else 1 if "last" in n else 2, -p.stat().st_mtime)
    cands.sort(key=rank)
    return cands[0] if cands else None

checkpoints = {}
for run, path in RUNS.items():
    ckpt = (SEED_DIR / path if not Path(path).is_absolute() else Path(path)) if path else pick_checkpoint(run)
    assert ckpt and ckpt.is_file(), f"Không tìm được checkpoint cho {run}; điền RUNS['{run}'] thủ công."
    checkpoints[run] = ckpt
    print(f"{run:18s} -> {ckpt}")

## 4. Load model + đánh giá (logic giống `src/evaluator.py`)

In [ ]:
from transformers import RTDetrConfig, RTDetrForObjectDetection, RTDetrImageProcessor
from ultralytics.utils.metrics import ap_per_class, box_iou

HF_MODEL_ID = "PekingU/rtdetr_r50vd"
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def is_ultralytics_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    return isinstance(ck, dict) and isinstance(ck.get("model") or ck.get("ema"), torch.nn.Module)

def load_hf_rtdetr(path, device):
    ck = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(ck, dict):
        for key in ("model_state", "state_dict"):
            if isinstance(ck.get(key), dict):
                ck = ck[key]; break
    assert isinstance(ck, dict) and all(torch.is_tensor(v) for v in ck.values()), f"{path} không phải tensor state-dict"
    sd = {}
    for k, v in ck.items():
        k2 = f"model.decoder.{k}" if k.startswith(("class_embed.", "bbox_embed.")) else k
        k2 = k2.replace("model.encoder.encoder.", "model.encoder.aifi.").replace(".self_attn.out_proj.", ".self_attn.o_proj.")
        if k2.startswith(("model.encoder.aifi.", "model.decoder.layers.")):
            k2 = re.sub(r"(?<!\.mlp)\.fc1\.", ".mlp.fc1.", k2)
            k2 = re.sub(r"(?<!\.mlp)\.fc2\.", ".mlp.fc2.", k2)
        if k2 in sd:
            assert torch.equal(sd[k2], v), f"Trọng số trùng nhưng khác nhau: {k2}"
            continue
        sd[k2] = v
    cfg = RTDetrConfig.from_pretrained(HF_MODEL_ID)
    cfg.num_labels = len(CLASS_NAMES)
    cfg.id2label = CLASS_NAMES
    cfg.label2id = {n: i for i, n in CLASS_NAMES.items()}
    model = RTDetrForObjectDetection(cfg)
    model.load_state_dict(sd, strict=True)
    processor = RTDetrImageProcessor.from_pretrained(HF_MODEL_ID)
    return model.to(device).eval(), processor

def read_labels(label_path, w, h):
    boxes, classes = [], []
    if label_path.is_file():
        for ln, line in enumerate(label_path.read_text(encoding="utf-8", errors="replace").splitlines(), 1):
            f = line.split()
            if not f:
                continue
            assert len(f) == 5, f"Label sai format {label_path}:{ln}"
            c, xc, yc, bw, bh = map(float, f)
            classes.append(int(c))
            boxes.append([(xc - bw/2)*w, (yc - bh/2)*h, (xc + bw/2)*w, (yc + bh/2)*h])
    return np.asarray(boxes, np.float32).reshape(-1, 4), np.asarray(classes, np.float32)

IOU_T = np.linspace(0.5, 0.95, 10)

def match(t_boxes, t_cls, p_boxes, p_cls):
    correct = np.zeros((len(p_boxes), len(IOU_T)), bool)
    if len(t_boxes) and len(p_boxes):
        ov = box_iou(torch.from_numpy(t_boxes), torch.from_numpy(p_boxes)).numpy()
        ov *= t_cls[:, None] == p_cls[None, :]
        for i, thr in enumerate(IOU_T):
            m = np.array(np.nonzero(ov >= thr)).T
            if len(m):
                if len(m) > 1:
                    m = m[np.argsort(ov[m[:, 0], m[:, 1]])[::-1]]
                    m = m[np.unique(m[:, 1], return_index=True)[1]]
                    m = m[np.unique(m[:, 0], return_index=True)[1]]
                correct[m[:, 1].astype(int), i] = True
    return correct

def summarize(stats, n_images, infer_s, model):
    tp, conf, pcls, tcls = (np.concatenate([s[i] for s in stats]) for i in range(4))
    _, _, p, r, _, ap, ap_cls, *_ = ap_per_class(tp, conf, pcls, tcls, names=CLASS_NAMES)
    per_class = {}
    for cid, name in CLASS_NAMES.items():
        idx = np.flatnonzero(ap_cls == cid)
        per_class[name] = {
            "ap50": float(ap[idx[0], 0]) if len(idx) else 0.0,
            "ap50_95": float(ap[idx[0]].mean()) if len(idx) else 0.0,
            "precision": float(p[idx[0]]) if len(idx) else 0.0,
            "recall": float(r[idx[0]]) if len(idx) else 0.0,
            "instances": int((tcls == cid).sum()),
        }
    P, R = float(p.mean()), float(r.mean())
    lat = infer_s / n_images * 1000
    return {
        "images": n_images, "instances": int(len(tcls)),
        "precision": P, "recall": R, "f1": 2*P*R/(P+R+1e-8),
        "map50": float(ap[:, 0].mean()), "map50_95": float(ap.mean()),
        "mean_5": float(np.mean([per_class[c]["ap50"] for c in DEFECT_CLASSES])),
        "latency_ms": lat, "fps": 1000/lat if lat else 0.0,
        "params_m": sum(x.numel() for x in model.parameters())/1e6,
    }, per_class

def evaluate_hf(ckpt, image_paths, label_dir):
    device = torch.device(DEVICE)
    model, processor = load_hf_rtdetr(ckpt, device)
    stats, infer_s = [], 0.0
    for b in range(0, len(image_paths), BATCH_SIZE):
        paths = image_paths[b:b+BATCH_SIZE]
        imgs, sizes, targets = [], [], []
        for p in paths:
            with Image.open(p) as im:
                im = im.convert("RGB")
            imgs.append(im); sizes.append(im.size)
            targets.append(read_labels(label_dir / f"{p.stem}.txt", *im.size))
        pv = processor(images=imgs, return_tensors="pt")["pixel_values"].to(device)
        if device.type == "cuda": torch.cuda.synchronize(device)
        t0 = time.perf_counter()
        with torch.inference_mode():
            out = model(pixel_values=pv)
        if device.type == "cuda": torch.cuda.synchronize(device)
        infer_s += time.perf_counter() - t0
        preds = processor.post_process_object_detection(
            out, threshold=0.0, target_sizes=torch.tensor([(h, w) for w, h in sizes], device=device))
        for (tb, tc), pr in zip(targets, preds):
            pb = pr["boxes"].cpu().numpy().astype(np.float32)
            pc = pr["labels"].cpu().numpy().astype(np.float32)
            sc = pr["scores"].cpu().numpy().astype(np.float32)
            stats.append((match(tb, tc, pb, pc), sc, pc, tc))
        done = min(b + BATCH_SIZE, len(image_paths))
        if done % (BATCH_SIZE * 50) < BATCH_SIZE or done == len(image_paths):
            print(f"  {done}/{len(image_paths)}")
    result = summarize(stats, len(image_paths), infer_s, model)
    del model; torch.cuda.empty_cache()
    return result

def evaluate_ultralytics(ckpt, run):
    from ultralytics import RTDETR
    import yaml
    yaml_path = OUT_DIR / f"_{run}_data.yaml"
    yaml_path.write_text(yaml.safe_dump({"path": str(DATA_ROOT), "train": f"images/{SPLIT}",
                                         "val": f"images/{SPLIT}", "names": CLASS_NAMES}), encoding="utf-8")
    m = RTDETR(str(ckpt))
    res = m.val(data=str(yaml_path), split="val", batch=BATCH_SIZE, imgsz=640, device=DEVICE,
                project=str(OUT_DIR), name=f"ultra_{run}", exist_ok=True, plots=False)
    per_class = {}
    for cid, name in CLASS_NAMES.items():
        idx = list(res.box.ap_class_index).index(cid) if cid in res.box.ap_class_index else None
        per_class[name] = {"ap50": float(res.box.ap50[idx]) if idx is not None else 0.0,
                           "ap50_95": float(res.box.ap[idx]) if idx is not None else 0.0}
    P, R = float(res.box.mp), float(res.box.mr)
    return {"images": None, "precision": P, "recall": R, "f1": 2*P*R/(P+R+1e-8),
            "map50": float(res.box.map50), "map50_95": float(res.box.map),
            "mean_5": float(np.mean([per_class[c]["ap50"] for c in DEFECT_CLASSES])),
            "latency_ms": float(res.speed["inference"]), "fps": 1000/float(res.speed["inference"])}, per_class

## 5. Chạy val

In [ ]:
OUT_DIR = SEED_DIR / "val_raw_results"
OUT_DIR.mkdir(exist_ok=True)

image_dir, label_dir = DATA_ROOT / "images" / SPLIT, DATA_ROOT / "labels" / SPLIT
image_paths = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMG_EXT)
if MAX_IMAGES:
    image_paths = image_paths[:MAX_IMAGES]
missing = sum(not (label_dir / f"{p.stem}.txt").is_file() for p in image_paths)
print(f"{len(image_paths)} ảnh {SPLIT} | {missing} ảnh không có file label (tính là background)")

summary_rows, class_rows = [], []
for run, ckpt in checkpoints.items():
    print(f"\n=== {run}  ({ckpt.name}) ===")
    backend = "ultralytics" if is_ultralytics_ckpt(ckpt) else "hf"
    overall, per_class = evaluate_ultralytics(ckpt, run) if backend == "ultralytics" else evaluate_hf(ckpt, image_paths, label_dir)
    summary_rows.append({"run": run, "dataset_eval": f"RAW/{SPLIT}", "backend": backend,
                         "checkpoint": str(ckpt.relative_to(SEED_DIR)), **overall})
    for name, v in per_class.items():
        class_rows.append({"run": run, "class": name, **v})
    print(f"  mAP50={overall['map50']:.4f}  mAP50-95={overall['map50_95']:.4f}  MEAN_5={overall['mean_5']:.4f}  "
          f"P={overall['precision']:.4f}  R={overall['recall']:.4f}  {overall['latency_ms']:.1f} ms/img")

summary = pd.DataFrame(summary_rows)
per_class_df = pd.DataFrame(class_rows)
tag = f"{SPLIT}" + (f"_first{MAX_IMAGES}" if MAX_IMAGES else "")
summary.to_csv(OUT_DIR / f"summary_{tag}.csv", index=False)
per_class_df.to_csv(OUT_DIR / f"per_class_{tag}.csv", index=False)
print("\nĐã lưu:", OUT_DIR)

## 6. Kết quả

In [ ]:
cols = ["run", "precision", "recall", "f1", "map50", "map50_95", "mean_5", "latency_ms", "fps"]
display(summary[cols].round(4))
display(per_class_df.pivot(index="class", columns="run", values="ap50").round(4)
        .rename(columns=lambda c: f"AP50 {c}"))
if len(summary) == 2:
    a, b = summary.set_index("run").loc[["raw_rtdetr50", "curated_rtdetr50"], ["map50", "map50_95", "mean_5", "f1"]].values
    print("curated - raw:", dict(zip(["map50", "map50_95", "mean_5", "f1"], np.round(b - a, 4))))